In [ ]:
%load_ext autoreload
%autoreload 3 -p

In [ ]:
import logging

import colorcet as cc
import holoviews as hv
import numpy as np
import pandas as pd
import xarray
from holoviews import opts
from joblib import delayed
from tqdm_joblib import ParallelPbar

import gnss_tid.center_finding
import gnss_tid.coords
import gnss_tid.fft
import gnss_tid.image
import gnss_tid.pointdata
import gnss_tid.spectral

hv.extension("bokeh")
opts.defaults(
    opts.Image(data_aspect=1, frame_height=300, frame_width=500, cmap=cc.cm.coolwarm, colorbar=True, show_grid=True, tools=["hover"]),
    hv.opts.Scatter(data_aspect=1, frame_height=300, frame_width=500, cmap=cc.cm.coolwarm, colorbar=True, show_grid=True, tools=["hover"]),
)
logging.basicConfig(level=logging.INFO)

In [ ]:
files = ["/disk1/tid/2015/0325/*.nc"]
lat_limits = (24.0, 48.0)
lon_limits = (-120.0, -75.0)
time_limits = ["20150325_234000", "20150325_235000"]

obs, rx = gnss_tid.pointdata.load_observations(
    files=files,
    lat_limits=lat_limits,
    lon_limits=lon_limits,
    time_limits=time_limits,
    el_min=30,
)

In [ ]:
proj = gnss_tid.pointdata.Local2D.from_geodetic(np.mean(lat_limits), np.mean(lon_limits), 300.0)
windows = gnss_tid.pointdata.make_time_windows(obs, 4, 2)
tobs = gnss_tid.pointdata.get_obs_in_window(obs, windows[4])
tobs = gnss_tid.pointdata.aggregate_by_receiver_satellite(tobs)
rxp = gnss_tid.pointdata.lookup_rx_location(tobs, rx)
def project(h):
    df = gnss_tid.pointdata.project_to_ipp(tobs, rxp.loc[:, gnss_tid.pointdata.LOOKUP_GEO].values, h)
    valid_lat = (df["lat"] > lat_limits[0]) & (df["lat"] < lat_limits[1])
    valid_lon = (df["lon"] > lon_limits[0]) & (df["lon"] < lon_limits[1])
    df = df.loc[valid_lat & valid_lon]
    df = gnss_tid.pointdata.convert_to_local_coords(df, proj)
    return df

In [ ]:
def plot(h, sv):
    df = project(h)
    return (
        hv.Points(df[df.sv == sv], kdims=["x", "y"], vdims=["dtec1", "sv"])
        .opts(
            color="dtec1",
            tools=[],
            frame_height=500,
            frame_width=800,
            clim=(-.5, .5),
            cmap=cc.cm.coolwarm,
            show_grid=True,
        )
    )

df = project(300)
SVS = df.sv.value_counts()[df.sv.value_counts() > 100].index.values
hv.output(widget_location='bottom')
hv.DynamicMap(plot, kdims=["height", "sv"]).redim.range(height=(100, 500)).redim.values(sv=SVS).opts(ylim=lat_limits, xlim=lon_limits)

In [ ]:
def plot(h):
    df = project(h)
    return (
        hv.Points(df, kdims=["x", "y"], vdims=["dtec1", "sv"])
        .opts(
            color="dtec1",
            tools=[],
            frame_height=500,
            frame_width=800,
            clim=(-.5, .5),
            cmap=cc.cm.coolwarm,
            show_grid=True,
        )
    )

hv.output(widget_location='bottom')
hv.DynamicMap(plot, kdims=["height"]).redim.range(height=(100, 500), x=(-2000, 2000), y=(-1500, 1500))